# Lab 04 Prelab Walkthrough — Sampling, Aliasing, and New Python Tools

Work through this notebook **before** your Lab 04 session, running every cell
and reading every explanation. It teaches this lab's *new* Python skills
using simulated signals — no hardware needed. The **CHECKPOINT** boxes ask
for specific values you will report in the **Prelab quiz on Canvas**.

Skills covered:

1. Simulating and *sampling* a sine wave in NumPy — seeing aliasing on your own screen
2. The folding formula and Python's `round()`
3. Reading numbers from the keyboard: `float(input(...))`
4. `while True:` loops and `break`
5. Pausing with `time.sleep` — and why it is not a precision timer
6. Multi-panel figures with `plt.subplots`

Run the setup cell first, as always:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 10

## 1. Sampling a sine wave — and your first alias

In lab you will sample real voltages, but the *mathematics* of sampling can
be explored right here. The trick: represent the "continuous" signal with a
very finely spaced time array, and the *sampled* signal with a coarse one.

The lab's standard test signal (manual Eq. 1) is
$y(t) = 2.5 + 2\sin(2\pi f t)$.

Let's generate a **9 Hz** signal and sample it at **fs = 10 Hz** — fewer
than two samples per cycle, so the Nyquist criterion is violated and we
should see an impostor.

In [ ]:
f_sig = 9.0          # signal frequency (Hz)
fs    = 10.0         # sample rate (Hz) — BELOW 2*f_sig: trouble ahead

t_fine = np.linspace(0, 1, 2000)             # pretend-continuous time
y_fine = 2.5 + 2*np.sin(2*np.pi*f_sig*t_fine)

n      = int(fs * 1.0)                       # samples in 1 second
t_samp = np.arange(n) / fs                   # sample times: 0, 1/fs, 2/fs, ...
y_samp = 2.5 + 2*np.sin(2*np.pi*f_sig*t_samp)  # what the DAQ would record

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(t_fine, y_fine, color='0.75', linewidth=1, label='true 9 Hz signal')
ax.plot(t_samp, y_samp, 'ro-', markersize=6, linewidth=1,
        label=f'samples at fs = {fs:.0f} Hz')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage (V)')
ax.legend()
ax.grid(linestyle='--', linewidth=0.5)
plt.show()

Look at the red curve the samples trace out: a slow, clean sine wave
that **does not exist** in the true signal. Nothing about the red data looks
"broken" — that is what makes aliasing dangerous.

`np.arange(n) / fs` is the new idiom here: the integers 0…n−1 divided by
the sample rate gives the sample times $t_k = k/f_s$. You will use it in
every capture script this lab.

> **CHECKPOINT 1:** From the plot, how many full cycles does the *sampled*
> (red) waveform complete in 1 second — i.e., what is the apparent
> frequency in Hz? Report this integer in the Canvas quiz.

## 2. The folding formula and `round()`

The apparent frequency you just counted is predicted by the manual's Eq. 3:

$$f_{apparent} = | f_{signal} - \mathrm{round}(f_{signal}/f_s)\cdot f_s |$$

Python's built-in `round()` rounds to the nearest whole number:

In [ ]:
print(round(0.3), round(0.9), round(1.125), round(2.6))

In [ ]:
def folding(f_signal, fs):
    """Apparent (alias) frequency after sampling f_signal at fs."""
    return abs(f_signal - round(f_signal / fs) * fs)

print(folding(9, 10))         # the case you just plotted
print(folding(100, 1000))     # below Nyquist: unchanged
print(folding(550, 1000))     # above Nyquist: folded!

Defining a small function (`def`, Lab 02) pays off immediately: you
will call `folding` for every prediction in lab.

> **CHECKPOINT 2:** Use `folding` to predict the apparent frequency when a
> **780 Hz** signal is sampled at **480 Hz**. Report the value in Hz.

In [ ]:
# CHECKPOINT 2 work area:


## 3. Reading numbers from the keyboard: `float(input(...))`

In lab, a script will pause while you read the handheld DMM, and you will
*type the reading in*. Lab 03's `input()` always hands back **text** — even
if you type digits — and text can't do math:

In [ ]:
typed = '3.302'              # what input() returns when you type 3.302
print(type(typed))

reading = float(typed)       # convert text -> number
print(type(reading), reading * 2)

So the lab pattern is `reading = float(input('DMM reading (V): '))`.
Try it live if you like (type a number and press Enter):

In [ ]:
# Uncomment and run to try it (a text box appears at the top of VS Code):
# reading = float(input('Type any decimal number: '))
# print(reading, 'doubled is', reading * 2)

> **CHECKPOINT 3:** What does `float('2.5') + 1` evaluate to, and what
> would `'2.5' + '1'` evaluate to instead? (Run them!) Report both answers.

In [ ]:
# CHECKPOINT 3 work area:
print(float('2.5') + 1)
print('2.5' + '1')

## 4. `while True:` loops and `break`

A `for` loop knows its itinerary in advance. But the flicker-fusion
staircase in Part-6 must keep going *until you say the flicker is gone* —
the stopping rule emerges inside the loop. That calls for `while True:`
with a `break`:

```python
while True:          # loop "forever"...
    ...
    if some_condition:
        break        # ...until this line runs — the ONLY exit
```

Let's simulate the staircase with a *virtual eye* whose fusion threshold is
picked randomly (seeded, so everyone gets the same one):

In [ ]:
rng = np.random.default_rng(42)
eye_threshold = rng.uniform(50, 80)      # the virtual eye's fusion frequency

freqs_tested = []
responses    = []

f = 20.0
while True:
    sees_flicker = f < eye_threshold     # the virtual eye answers
    answer = 'y' if sees_flicker else 'n'
    print(f"{f:.0f} Hz — flicker? {answer}")

    freqs_tested.append(f)
    responses.append(1 if answer == 'y' else 0)

    if answer != 'y':
        break                            # fusion reached — stop
    f = f + 5                            # step up and ask again

print(f"\nThreshold is between {freqs_tested[-2]:.0f} and {freqs_tested[-1]:.0f} Hz")

In lab, the `sees_flicker` line is replaced by `input(...)` — *you*
are the eye. Everything else is identical: this loop is 90% of your Part-6
code.

**Safety habit:** before running any `while True:` loop, point at the
`break` and confirm it is reachable. A loop with no reachable `break` runs
forever (Interrupt button to the rescue).

> **CHECKPOINT 4:** At what frequency (in Hz) does the staircase above
> stop — i.e., the first frequency where the virtual eye reports no
> flicker? Report the integer.

## 5. `time.sleep` — the honest, imprecise pause

`time.sleep(x)` pauses the script for about `x` seconds. In lab it gives
the Wavegen time to settle, and it sets the (rough) blink rate of your
DIO-driven LED. How rough? Measure it:

In [ ]:
import time

t0 = time.perf_counter()          # a high-resolution stopwatch
for k in range(10):
    time.sleep(0.05)              # request: 10 x 0.05 s = 0.500 s total
elapsed = time.perf_counter() - t0

print(f"requested 0.500 s, got {elapsed:.4f} s "
      f"({(elapsed - 0.5)*1000:.1f} ms over)")

Typically you get slightly *more* than requested — the operating
system wakes your script up when it gets around to it. A few ms of slop is
invisible in a 2 Hz blink and fatal in a 100 Hz signal. That is why lab
Part-6 drives the flicker LED from the **Wavegen** (hardware timing inside
the ADS) instead of a `sleep` loop.

*(No checkpoint — your measured slop is your machine's business.)*

## 6. Multi-panel figures with `plt.subplots`

Your aliasing deliverable is a **five-panel** figure. `plt.subplots(rows,
cols)` returns one figure and an *array* of axes — and everything you know
about formatting one `ax` applies to each. Loop over them with `zip`:

In [ ]:
f_sig = 9.0
rates = [100, 12, 10]                    # three sample rates to compare

fig, axes = plt.subplots(3, 1, figsize=(6.5, 7.0))

for ax, fs in zip(axes, rates):
    n      = int(fs * 1.0)
    t_samp = np.arange(n) / fs
    y_samp = 2.5 + 2*np.sin(2*np.pi*f_sig*t_samp)

    ax.plot(t_samp, y_samp, 'ro-', markersize=4, linewidth=0.8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Voltage (V)')
    ax.set_title(f'9 Hz signal sampled at fs = {fs} Hz '
                 f'(predicted apparent: {folding(f_sig, fs):.0f} Hz)',
                 fontsize=10)
    ax.grid(linestyle='--', linewidth=0.5)

fig.tight_layout()
plt.show()

Study how one loop pass builds one complete panel — the lab's
five-panel version is this exact pattern with five CSV files.

> **CHECKPOINT 5:** In the *bottom* panel (fs = 10 Hz), what apparent
> frequency does the folding formula predict — and does the plot agree?
> Report the predicted value in Hz.

## Summary: where each skill is used in lab

| Skill | Where you will use it |
|---|---|
| `np.arange(n) / fs` sample times | every capture script (Parts 3–5) |
| `folding(f_signal, fs)` / `round()` | predicting every Part-4 and Part-5 result |
| `float(input(...))` | typing DMM readings into the Part-3 script |
| `while True:` + `break` | the Part-6 flicker-fusion staircase |
| `time.sleep` | Wavegen settling (Part-4), DIO blinking (Part-5) |
| `plt.subplots(5, 1)` + axes loop | the five-panel aliasing deliverable |

See you in lab — where the signals are real, and one of them will lie to
you.